In [ ]:
import tkinter as tk
import pygame
from pygame.locals import *
import os
from tkinter import messagebox
from functools import partial

# Инициализация Pygame
pygame.init()


WIDTH = 400
HEIGHT = 400 # Размеры окна


WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
BLUE = (0, 0, 255) # Цвета



screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Шахматная доска")

clock = pygame.time.Clock()



N = 0
L = 0
K = 0

# Список для хранения координат фигур
chess_pieces = []

# Функция для чтения координат фигур из файла
def read_chess_pieces_from_file(file_path):
    pieces = []
    try:
        with open(file_path, 'r') as file:
            for line in file:
                x, y = map(int, line.strip().split())
                pieces.append((x, y))
    except FileNotFoundError:
        print(f"Файл '{file_path}' не найден.")
    return pieces

# Функция для отображения шахматной доски
def draw_chessboard():
    square_size = WIDTH // N

    for i in range(N):
        for j in range(N):
            if (i + j) % 2 == 0:
                pygame.draw.rect(screen, WHITE, (i * square_size, j * square_size, square_size, square_size))
            else:
                pygame.draw.rect(screen, BLACK, (i * square_size, j * square_size, square_size, square_size))

# Функция для отображения фигур на доске
def draw_chess_pieces():
    square_size = WIDTH // N
    for piece in chess_pieces:
        x, y = piece
        pygame.draw.circle(screen, RED, (x * square_size + square_size // 2, y * square_size + square_size // 2), square_size // 2)

# Функция для создания нового окна
def create_new_window():
    global N, L, K, chess_pieces

    # Получение значений из полей ввода
    N = int(entry_N.get())
    L = int(entry_L.get())
    K = int(entry_K.get())

    # Чтение координат фигур из файла
    chess_pieces = read_chess_pieces_from_file("input.txt")

    # Закрытие главного окна
    root.destroy()

    # Создание нового окна Tkinter
    new_window = tk.Toplevel()
    new_window.title("Шахматная доска")

    # Создание холста для Pygame
    frame = tk.Frame(new_window, width=WIDTH, height=HEIGHT)
    frame.pack()

    # Инициализация Pygame на холсте
    os.environ['SDL_WINDOWID'] = str(frame.winfo_id())
    pygame.display.init()
    screen = pygame.display.set_mode((WIDTH, HEIGHT))

    # Функция для отображения шахматной доски и фигур
    def draw_canvas():
        screen.fill(BLUE)
        draw_chessboard()
        draw_chess_pieces()
        pygame.display.flip()

    # Отображение шахматной доски и фигур
    draw_canvas()

    # Функция для сохранения решения в файл
    def save_solution_to_file():
        if len(chess_pieces) == K:
            with open("output.txt", 'a') as file:
                file.write(" ".join([f"({x},{y})" for x, y in chess_pieces]) + "\n")
            messagebox.showinfo("Решение сохранено", "Решение сохранено в файл 'output.txt'.")
        else:
            with open("output.txt", 'a') as file:
                file.write("no solution\n")
            messagebox.showinfo("Решение не найдено", "Решений не найдено. Записано 'no solution' в файл 'output.txt'.")
        new_window.destroy()

    # Функция для обработки закрытия окна
    def handle_window_close():
        if len(chess_pieces) == K:
            response = messagebox.askquestion("Добавить решение?", "Решение найдено. Хотите добавить его в файл 'output.txt'?")
            if response == "yes":
                save_solution_to_file()
        else:
            with open("output.txt", 'a') as file:
                file.write("no solution\n")
            messagebox.showinfo("Решение не найдено", "Решений не найдено. Записано 'no solution' в файл 'output.txt'.")
            new_window.destroy()

    # Привязка функции handle_window_close() к событию закрытия окна
    new_window.protocol("WM_DELETE_WINDOW", handle_window_close)

    # Обновление окна при изменении размера
    def on_resize(event, width, height):
        screen = pygame.display.set_mode((width, height))
        draw_canvas()

    new_window.bind("<Configure>", lambda event: on_resize(event, event.width, event.height))

    # Создание кнопки для сохранения решения в файл
    save_button = tk.Button(new_window, text="Добавить решение", command=save_solution_to_file)
    save_button.pack()

    # Запуск главного цикла Tkinter
    new_window.mainloop()

# Создание главного окна Tkinter
root = tk.Tk()
root.title("Шахматные фигуры")

# Создание полей ввода
label_N = tk.Label(root, text="Размер доски (N):")
label_N.pack()
entry_N = tk.Entry(root)
entry_N.pack()

label_L = tk.Label(root, text="Количество фигур (L):")
label_L.pack()
entry_L = tk.Entry(root)
entry_L.pack()

label_K = tk.Label(root, text="Количество стоящих фигур (K):")
label_K.pack()
entry_K = tk.Entry(root)
entry_K.pack()

# Создание кнопки
button = tk.Button(root, text="Создать новое окно", command=create_new_window)
button.pack()

# Запуск главного цикла Tkinter
root.mainloop()